# Model Training

## Imports

In [ ]:
from ultralytics import YOLO
from onnxruntime.quantization import QuantType, quantize_dynamic
import torch
import os
import random
import shutil
from pathlib import Path

## Check GPU availability

In [ ]:
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
else:
    print("No GPU detected")

## Splitting into train/valid/test folders

In [ ]:
dataset_root = Path("PAPI_Night.yolo26")

random.seed(42)


def find_images_and_labels(root):
    # case 1: structured dataset
    if (root / "train" / "images").exists():
        img_dir = root / "train" / "images"
        lbl_dir = root / "train" / "labels"
    else:
        # fallback: flat train folder
        img_dir = root / "train"
        lbl_dir = root / "train"

    return img_dir, lbl_dir


def get_pairs(img_dir, lbl_dir):
    exts = [".jpg", ".jpeg", ".png"]

    label_map = {p.stem.lower(): p for p in lbl_dir.glob("*.txt")}
    pairs = []

    for img in img_dir.iterdir():
        if img.suffix.lower() in exts:
            key = img.stem.lower()

            if key in label_map:
                pairs.append((img, label_map[key]))

    return pairs


def split(pairs):
    random.shuffle(pairs)

    n = len(pairs)
    train_end = int(n * 0.7)
    val_end = train_end + int(n * 0.2)

    return (
        pairs[:train_end],
        pairs[train_end:val_end],
        pairs[val_end:]
    )


def move(pairs, split_name, root):
    img_out = root / split_name / "images"
    lbl_out = root / split_name / "labels"

    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img, lbl in pairs:
        shutil.move(str(img), img_out / img.name)
        shutil.move(str(lbl), lbl_out / lbl.name)


def run():
    img_dir, lbl_dir = find_images_and_labels(dataset_root)

    print("Using:")
    print("Images:", img_dir)
    print("Labels:", lbl_dir)

    pairs = get_pairs(img_dir, lbl_dir)

    if not pairs:
        raise ValueError("No pairs found — check dataset structure")

    train, val, test = split(pairs)

    move(train, "train", dataset_root)
    move(val, "valid", dataset_root)
    move(test, "test", dataset_root)

    print("Total:", len(pairs))
    print("Train:", len(train))
    print("Val:", len(val))
    print("Test:", len(test))


run()

## Model training

In [ ]:
def train():
    model = YOLO("yolo26n.pt")

    model.train(
        data="PAPI_Night.yolo26\data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        workers=4,
        patience=15,
        save=True,
        # colour-safe aug: Ultralytics colour-jitter defaults (hsv_s=0.7, mosaic=1.0)
        # can swap red<->white<->transition while labels stay fixed; no h-flip (lamp order)
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, mosaic=0.0, mixup=0.0, copy_paste=0.0,
        erasing=0.0, degrees=0.0, fliplr=0.5, flipud=0.0,
    )

train()

In [ ]:
def evaluate():
    model = YOLO("runs/detect/train-2/weights/best.pt")

    metrics = model.val(data="PAPI_Night.yolo26\data.yaml")

    print(metrics)

evaluate()

In [ ]:
def export():
    model = YOLO("runs/detect/train-2/weights/best.pt")

    model.export(
        format="onnx",
        # imgsz MUST match the exported checkpoint's training size — serving model is 1280, not 640
        imgsz=1280,
        dynamic=True
    )

export()

## Edge optimization

In [ ]:
def quantize_model():
    quantize_dynamic(
        model_input=r"runs\detect\train-2\weights\best.onnx",
        model_output="best_int8.onnx",
        weight_type=QuantType.QInt8,  # was 3 (== QInt16, NOT int8)
    )  # NOTE: dynamic quant is weights-only + UNCALIBRATED; prefer static int8 w/ a calibration set

quantize_model()

## Training YOLO26s (s=small) model for comparison

In [ ]:
def train():
    model = YOLO("yolo26s.pt")

    model.train(
        data="PAPI_Night.yolo26\data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        workers=4,
        patience=15,
        save=True,
        # colour-safe aug: Ultralytics colour-jitter defaults (hsv_s=0.7, mosaic=1.0)
        # can swap red<->white<->transition while labels stay fixed; no h-flip (lamp order)
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, mosaic=0.0, mixup=0.0, copy_paste=0.0,
        erasing=0.0, degrees=0.0, fliplr=0.5, flipud=0.0,
    )

train()

In [ ]:
def evaluate():
    model = YOLO("runs/detect/train-3/weights/best.pt")

    metrics = model.val(data="PAPI_Night.yolo26\data.yaml")

    print(metrics)

evaluate()

## Training on augmented dataset

In [ ]:
def train():
    model = YOLO("yolo26s.pt")

    model.train(
        data="PAPI_Night_Augmented.yolo26\data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        workers=4,
        patience=15,
        save=True,
        # colour-safe aug: Ultralytics colour-jitter defaults (hsv_s=0.7, mosaic=1.0)
        # can swap red<->white<->transition while labels stay fixed; no h-flip (lamp order)
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, mosaic=0.0, mixup=0.0, copy_paste=0.0,
        erasing=0.0, degrees=0.0, fliplr=0.5, flipud=0.0,
    )

train()

In [ ]:
def evaluate():
    model = YOLO("runs/detect/train-5/weights/best.pt")

    metrics = model.val(data="PAPI_Night_Augmented.yolo26\data.yaml")

    print(metrics)

evaluate()

## Training on full dataset

In [ ]:
def train():
    model = YOLO("yolo26s.pt")

    model.train(
        data="PAPI_Split\data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        workers=4,
        patience=15,
        save=True,
        # colour-safe aug: Ultralytics colour-jitter defaults (hsv_s=0.7, mosaic=1.0)
        # can swap red<->white<->transition while labels stay fixed; no h-flip (lamp order)
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, mosaic=0.0, mixup=0.0, copy_paste=0.0,
        erasing=0.0, degrees=0.0, fliplr=0.5, flipud=0.0,
    )

train()

In [ ]:
def evaluate():
    model = YOLO("runs/detect/train-6/weights/best.pt")

    metrics = model.val(data="PAPI_Split\data.yaml")

    print(metrics)

evaluate()

## Training 1280x1280 model instead of 640x640 for comparison

In [ ]:
def train():
    model = YOLO("yolo26s.pt")

    model.train(
        data="PAPI_Split\data.yaml",
        epochs=100,
        imgsz=1280,
        batch=4,
        device=0,
        workers=1,
        patience=15,
        save=True,
        # colour-safe aug: Ultralytics colour-jitter defaults (hsv_s=0.7, mosaic=1.0)
        # can swap red<->white<->transition while labels stay fixed; no h-flip (lamp order)
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, mosaic=0.0, mixup=0.0, copy_paste=0.0,
        erasing=0.0, degrees=0.0, fliplr=0.5, flipud=0.0,
    )

train()

In [ ]:
def evaluate():
    model = YOLO("runs/detect/train-7/weights/best.pt")

    metrics = model.val(data="PAPI_Split\data.yaml")

    print(metrics)

evaluate()